# Agentic AI Hazard and Risk Orchestrator Demo

This demo supports the assignment **Designing an Agentic AI Workflow for Hazard and Risk Decision Support**.

You can use it in two ways:

1. **No API key:** Run the notebook to generate a complete prompt, then copy/paste the prompt into Gemini in a browser.
2. **With Gemini API key:** Add a Gemini API key in Colab Secrets and run the optional API cell.

This is a classroom simulation. Do not enter private, sensitive, or real emergency-response information.

## Step 1: Choose scenario mode

You may use either:

- **Option A:** Imagined classroom scenario.
- **Option B:** A simple source-based scenario using a public source such as FEMA National Risk Index for one location.

For Option B, students can use the FEMA NRI website or a simple exported table to fill in agent findings. This notebook uses a sample scenario so it can run without downloading external data.

In [ ]:
import pandas as pd
from textwrap import dedent

scenario_mode = "Option A: Imagined classroom scenario"
location = "Example County"
hazard = "River flooding after several days of heavy rainfall"

scenario_description = (
    "A county emergency management team is preparing for possible river flooding "
    "after several days of heavy rainfall. The county includes river-adjacent "
    "neighborhoods, rural roads, several schools, one hospital, and communities "
    "with different levels of vulnerability and response capacity."
)

print("Scenario mode:", scenario_mode)
print("Location:", location)
print("Hazard:", hazard)
print("Scenario:", scenario_description)

## Step 2: Create sample agent cards

Each agent represents one vertical data layer. Students can edit the text below to match their own location and hazard.

In [ ]:
agent_cards = [
    {
        "Agent name": "Hazard Agent",
        "Purpose": "Identify the main hazard concern.",
        "Data layer used": "Flood history, rainfall forecast, river floodplain maps, stream gauge trends.",
        "Question answered": "Where is flooding most likely or most concerning?",
        "Finding": "The main hazard concern is river flooding in low-lying areas near major streams. Heavy rainfall may increase river levels and create localized flooding in floodplain areas.",
        "Possible limitation": "The finding is based on general flood-prone areas and does not include the latest verified river-stage measurements.",
        "Human review needed": "Emergency managers should check current stream gauge data and local flood reports."
    },
    {
        "Agent name": "Exposure Agent",
        "Purpose": "Identify what people, infrastructure, and assets may be in harm's way.",
        "Data layer used": "Buildings, roads, schools, hospital location, utilities, population distribution.",
        "Question answered": "What could be affected if flooding occurs?",
        "Finding": "Several residential areas, local roads, two schools, and one hospital access route are near areas that may be affected by flooding.",
        "Possible limitation": "The finding does not confirm whether buildings or roads are currently flooded.",
        "Human review needed": "Local officials should verify current road and facility status."
    },
    {
        "Agent name": "Vulnerability Agent",
        "Purpose": "Identify communities or systems that may be more sensitive to harm.",
        "Data layer used": "Older adult population, vehicle access, income, housing condition, disability, language access.",
        "Question answered": "Which communities may need extra support?",
        "Finding": "Some neighborhoods near the river have higher shares of older residents and households with limited vehicle access. These areas may need additional communication or evacuation support.",
        "Possible limitation": "Vulnerability data may be outdated or may not capture local informal support networks.",
        "Human review needed": "Community partners should confirm whether the priority areas match local knowledge."
    },
    {
        "Agent name": "Resilience and Capacity Agent",
        "Purpose": "Identify available response and recovery resources.",
        "Data layer used": "Shelter locations, emergency services, road network, evacuation routes, hospital access, communication systems.",
        "Question answered": "What resources are available, and where are response gaps possible?",
        "Finding": "The county has shelters and emergency services available, but some evacuation routes may overlap with flood-prone roads. Backup routes and shelter accessibility should be checked.",
        "Possible limitation": "The finding does not include current shelter capacity or real-time road closure information.",
        "Human review needed": "Emergency managers should confirm shelter status, road closures, and resource availability."
    },
    {
        "Agent name": "Situational Awareness Agent",
        "Purpose": "Identify current observations that should be checked before decisions are made.",
        "Data layer used": "Weather alerts, stream gauges, traffic reports, emergency calls, field reports, social media, drone or satellite imagery.",
        "Question answered": "What current information is needed now?",
        "Finding": "Current rainfall totals, stream gauge readings, road closure reports, and field observations should be checked before making final decisions.",
        "Possible limitation": "Crowdsourced reports may be incomplete, duplicated, or inaccurate.",
        "Human review needed": "Field staff should confirm conditions before public decisions are made."
    }
]

agents_df = pd.DataFrame(agent_cards)
agents_df

## Step 3: Build the orchestrator prompt

This prompt can be pasted into Gemini, ChatGPT, Microsoft Copilot, NotebookLM, or another approved chatbot.

In [ ]:
def build_orchestrator_prompt(scenario_mode, location, hazard, scenario_description, agent_cards):
    findings_text = "\n".join([f"{a['Agent name']}: {a['Finding']}" for a in agent_cards])
    prompt = f"""
You are acting as a disaster-risk AI orchestrator for a county emergency management team. Your job is to combine the findings from several specialized agents into one short decision-support summary.

Do not invent exact numbers. Use only the information provided below. If information is missing, identify what should be checked next. This is a classroom simulation, not a real emergency decision.

Scenario mode: {scenario_mode}
Location: {location}
Hazard: {hazard}
Scenario: {scenario_description}

Agent findings:
{findings_text}

Create a decision-support summary with the following sections:
1. Overall situation summary
2. Priority concerns
3. Why these concerns matter
4. Recommended next data checks
5. Human review needed before action
6. Main limitations of the AI-assisted summary
7. One sentence explaining how this agentic AI workflow is different from a single forecasting model
"""
    return dedent(prompt).strip()

orchestrator_prompt = build_orchestrator_prompt(scenario_mode, location, hazard, scenario_description, agent_cards)
print(orchestrator_prompt)

## Step 4A: No API key option

Copy the prompt printed above and paste it into Gemini in a browser. Then copy the response into your submission sheet.

## Step 4B: Optional Gemini API call from Colab

This step requires a Gemini API key. In Colab, open **Secrets** in the left sidebar and add a key named `GEMINI_API_KEY`.

If this cell fails because of access, quota, or model availability, use the copy/paste browser option instead.

In [ ]:
# Optional Gemini API call. Run only if you have a Gemini API key.

USE_GEMINI_API = False  # Change to True if you have added GEMINI_API_KEY in Colab Secrets.

if USE_GEMINI_API:
    !pip install -q google-genai
    from google import genai
    from google.colab import userdata

    api_key = userdata.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("No GEMINI_API_KEY found in Colab Secrets. Use the copy/paste option instead.")

    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=orchestrator_prompt
    )
    print(response.text)
else:
    print("USE_GEMINI_API is False. Copy the prompt above and paste it into Gemini in a browser.")

## Step 5: Generate a visualization prompt

Ask the chatbot to create a Mermaid flowchart. Mermaid is useful because students can submit the code or render it as a diagram.

In [ ]:
mermaid_prompt = """
Create a Mermaid flowchart for this agentic AI disaster-risk workflow.

Use this structure:
- Hazard data goes to Hazard Agent
- Exposure data goes to Exposure Agent
- Vulnerability data goes to Vulnerability Agent
- Resilience/capacity data goes to Resilience and Capacity Agent
- Situational awareness data goes to Situational Awareness Agent
- All five agents send their findings to a central Orchestrator Agent
- The Orchestrator Agent produces: priority concerns, recommended next data checks, human review before action, and main limitations

Return only valid Mermaid code using flowchart TD. Keep labels short and readable.
""".strip()

print(mermaid_prompt)

In [ ]:
# Example Mermaid code if students need a starting point.
example_mermaid = """
flowchart TD
    A[Hazard Data] --> B[Hazard Agent]
    C[Exposure Data] --> D[Exposure Agent]
    E[Vulnerability Data] --> F[Vulnerability Agent]
    G[Resilience / Capacity Data] --> H[Resilience Agent]
    I[Situational Awareness Data] --> J[Situational Awareness Agent]

    B --> K[Orchestrator Agent]
    D --> K
    F --> K
    H --> K
    J --> K

    K --> L[Priority Concerns]
    K --> M[Recommended Data Checks]
    K --> N[Human Review Before Action]
    K --> O[Main Limitations]
""".strip()

print(example_mermaid)

## Step 6: Save files for submission

This cell saves the agent cards, prompt, and Mermaid code as files. You can download them from the Colab file browser.

In [ ]:
agents_df.to_csv("agent_cards_and_findings.csv", index=False)
with open("orchestrator_prompt.txt", "w") as f:
    f.write(orchestrator_prompt)
with open("example_mermaid_diagram.txt", "w") as f:
    f.write(example_mermaid)

print("Saved:")
print("- agent_cards_and_findings.csv")
print("- orchestrator_prompt.txt")
print("- example_mermaid_diagram.txt")

## Optional source-based version using FEMA National Risk Index

Students who choose an actual location can use FEMA National Risk Index or a similar public source to fill in the agent cards. A simple approach is:

1. Go to the FEMA National Risk Index website.
2. Search for one county or community.
3. Record the main hazard concern, risk rating, expected annual loss information, social vulnerability, and community resilience.
4. Use those notes to revise the Hazard, Exposure, Vulnerability, and Resilience agent findings.

No full data analysis is required unless the instructor asks for it.